# Energy demand models for industrial thermal processes – Python implementation

Python-Umsetzung der **reinen Energiebedarfsmodelle** aus *Manuscript_final.docx*. **Aktuell implementiert: Pasteurisierung** (Eq. 18, 19). Pinch-Analyse, GCC und Wärmerückgewinnung sind nicht Teil des Notebooks, sie laufen im Gesamtsystem. Das Notebook liefert dafür die Stromdaten (`result.stream_table()`). Weitere Prozesse sind in der Prozessauswahl bereits angelegt (Status *planned*).

**Aufbau**
1. Setup und Framework (`InputSpec`, Prozess-Registry, Input-Prüfung mit Defaults)
2. Pasteurisierungsmodell (Eq. 18/19)
3. Visualisierung inkl. vereinfachtem P&ID
4. Registrierung der Prozesse
5. Beispielrechnung, Validierung gegen das Manuscript, **Vergleich mit `classes.py`** (Abschnitt 7), interaktive Oberfläche

**Eingaben:** *required* = Pflicht (Fehlermeldung, wenn es fehlt). *optional* = Default wird genutzt, in der Ergebnistabelle steht, woher jeder Wert kommt (`user input` / `default` / `preset`).

**Neuen Prozess hinzufügen:** `InputSpec`-Liste + `model(values) -> result` + Plot-Funktionen schreiben und mit `register_process(...)` eintragen. Die Oberfläche baut sich daraus automatisch auf.

| Manuscript | Umsetzung im Code |
|---|---|
| Eq. (18) Heizbedarf Pasteurisierung | `pasteurization_model` → `Q_heating` |
| Eq. (19) Kühlbedarf | `pasteurization_model` → `Q_cooling` |
| Fig. 7 Schema Pasteurisierung | `draw_pid` |
| Eq. (22) RMSE | Validierungszelle |

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyBboxPatch, Polygon

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

## 1  Framework: Inputs, Prozess-Registry, Defaults

In [ ]:
@dataclass(frozen=True)
class InputSpec:
    """Description of ONE model input. Drives validation, defaults, audit table and the UI form."""
    key: str                          # python argument name
    label: str                        # human-readable description
    unit: str
    required: bool                    # True = must be given, False = default is used if missing
    symbol: str = ""                  # symbol as used in the manuscript
    default: Optional[object] = None  # default for optional inputs (None = derived, see derive_defaults)
    example: Optional[object] = None  # pre-filled value in the UI
    kind: str = "float"               # "float" | "choice"
    choices: tuple = ()
    ref: str = ""                     # where the input is introduced in the manuscript
    note: str = ""


@dataclass
class ProcessModel:
    """One entry of the process registry (= one selectable process in the UI)."""
    key: str
    name: str
    process_type: str                 # level 1 of manuscript Fig. 1
    category: str                     # level 2 of manuscript Fig. 1
    status: str                       # "implemented" | "planned"
    manuscript_refs: str
    inputs: list = field(default_factory=list)
    run: Optional[Callable] = None            # run(values: dict) -> result object
    derive_defaults: Optional[Callable] = None  # values -> {key: (value, source_text)}
    figures: list = field(default_factory=list)  # [(title, fn(result) -> Figure)]
    planned_inputs_text: str = ""


PROCESS_REGISTRY: dict[str, ProcessModel] = {}


def register_process(model: ProcessModel) -> ProcessModel:
    PROCESS_REGISTRY[model.key] = model
    return model


def resolve_inputs(model: ProcessModel, user_values: dict):
    """Check required inputs, fill optional ones with defaults and return (values, audit_table)."""
    specs = {s.key: s for s in model.inputs}
    unknown = set(user_values) - set(specs)
    if unknown:
        raise KeyError(f"Unknown input(s) for '{model.key}': {sorted(unknown)}")

    values, source, missing = {}, {}, []
    for s in model.inputs:
        v = user_values.get(s.key)
        if v is None:
            if s.required:
                missing.append(s.key)
                continue
            values[s.key] = s.default
            source[s.key] = "default" if s.default is not None else "derived default"
        else:
            if s.kind == "float":
                v = float(v)
            elif s.kind == "choice" and v not in s.choices:
                raise ValueError(f"'{s.key}' must be one of {list(s.choices)}, got {v!r}")
            values[s.key] = v
            source[s.key] = "user input"
    if missing:
        raise ValueError(f"Missing required input(s): {missing}")

    if model.derive_defaults is not None:
        for k, (val, src) in model.derive_defaults(values).items():
            if values.get(k) is None:
                values[k], source[k] = val, src

    audit = pd.DataFrame([{
        "Symbol": s.symbol, "Input": s.label, "Value": values[s.key], "Unit": s.unit,
        "Type": "required" if s.required else "optional", "Source": source[s.key],
        "Manuscript / note": " | ".join(x for x in (s.ref, s.note) if x),
    } for s in model.inputs])
    return values, audit

## 2  Pasteurisierung – Energiebedarfsmodell
**Manuscript:** Abschnitt *Continuous heating of fluids*

$$Q_{heating}=\dot m\,c_p\,(T_{pasteur}-T_{in})+\frac1t\sum_i A\,U\,(T_i-T_{amb})\quad(18)$$
$$Q_{cooling}=\dot m\,c_p\,(T_{pasteur}-T_{out})-\frac1t\sum_i A\,U\,(T_i-T_{amb})\quad(19)$$

**Pflicht-Inputs** (laut Manuscript): $\dot m$, $T_{in}$, $T_{pasteur}$, $T_{out}$ – und im Manuscript auch $c_p$.
**Hier optional:** $c_p$ (Preset je Produkt), Sicherheitsaufschlag auf $T_{pasteur}$, Verlustterm ($A$, $U$, $T_{amb}$).

**Annahmen, die das Manuscript offen lässt (bitte prüfen):**
- $T_i$ im Verlustterm ist nicht definiert → hier: Mittelwert der Stufe (Eintritt/Austritt), entspricht dem Zeitmittel bei linearem Profil. Verluste = 0 per Default.
- Die Haltezeit geht nicht in den Energiebedarf ein (so auch im Manuscript).

**Übergabe an die Pinch-Analyse:** `result.stream_table()` gibt den kalten Strom (Aufheizen) und den heißen Strom (Abkühlen) mit Start-/Zieltemperatur, CP und Leistung aus. Die Spaltennamen ggf. an dein System anpassen.

In [ ]:
# ---------------------------------------------------------------------------------------
# Default specific heat capacities  [kJ/(kg K)]
# NOTE: typical values, NOT taken from the manuscript -> please check/replace.
# ---------------------------------------------------------------------------------------
PRODUCT_CP_PRESETS = {
    "Juice (generic)": 3.89,   # = 3890 J/(kg K) from the example in main.py
    "Whole milk": 3.93,
    "Skim milk": 3.97,
    "Water": 4.18,
}

_REF_IN = "Sect. 'Continuous heating of fluids', input list"

PASTEURIZATION_INPUTS = [
    # ---- required (manuscript: input list below Eq. 19) -----------------------------------
    InputSpec("mass_flow_kg_h", "Product mass flow rate", "kg/h", True, "m_product",
              example=3135.0, ref=_REF_IN),
    InputSpec("T_in", "Product inlet temperature", "degC", True, "T_in",
              example=4.0, ref=_REF_IN),
    InputSpec("T_pasteur", "Pasteurization temperature", "degC", True, "T_pasteur",
              example=90.0, ref=_REF_IN),
    InputSpec("T_out", "Outlet temperature after cooling", "degC", True, "T_out",
              example=4.0, ref=_REF_IN),
    # ---- optional --------------------------------------------------------------------------
    InputSpec("product", "Product (selects default cp)", "-", False, "",
              default="Juice (generic)", kind="choice", choices=tuple(PRODUCT_CP_PRESETS),
              note="Only used to pick a default cp if cp is not given."),
    InputSpec("cp", "Specific heat capacity of product", "kJ/(kg K)", False, "cp,product",
              default=None, example=3.89, ref=_REF_IN,
              note="Required in the manuscript; here optional -> preset of the selected product."),
    InputSpec("T_safety_margin", "Safety margin above pasteurization temperature", "K", False,
              "dT_margin", default=0.0, example=0.0, ref="Sect. 'Continuous heating of fluids' (1st paragraph)",
              note="Manuscript: temperature 'slightly exceeded', not quantified -> 0 K."),
    InputSpec("loss_area", "Heat-losing surface (pipes, holding tube, exchangers)", "m2", False,
              "A_pasteur", default=0.0, example=0.0, ref="Eq. (18), (19) loss term",
              note="0 = no heat losses (as in the validation case)."),
    InputSpec("U_value", "Overall heat transfer coefficient to ambient", "W/(m2 K)", False,
              "U_pasteur", default=1.0, example=1.0, ref="Eq. (18), (19) loss term",
              note="Dummy value; only effective if loss_area > 0."),
    InputSpec("T_amb", "Ambient temperature", "degC", False, "T_amb",
              default=20.0, example=20.0, ref="Eq. (18), (19) loss term"),
]


def pasteurization_derive_defaults(values: dict) -> dict:
    """Derived defaults: cp from product preset."""
    p = values["product"]
    return {"cp": (PRODUCT_CP_PRESETS[p], f"preset ({p}), not from manuscript")}


@dataclass
class PasteurizationResult:
    inputs: dict
    T_pasteur_eff: float           # pasteurization temperature incl. safety margin [degC]
    # Eq. (18)
    Q_sensible_heating: float
    Q_loss_heating: float
    Q_heating: float
    # Eq. (19)
    Q_sensible_cooling: float
    Q_loss_cooling: float
    Q_cooling: float
    CP_flow: float                 # m_dot * cp [kW/K]
    warnings: list = field(default_factory=list)

    def summary(self) -> pd.DataFrame:
        rows = [
            ("Heating demand, sensible", "m*cp*(T_pasteur - T_in)", self.Q_sensible_heating, "kW", "Eq. (18), 1st term"),
            ("Heat losses during heating", "(1/t) sum A*U*(T_i - T_amb)", self.Q_loss_heating, "kW", "Eq. (18), 2nd term"),
            ("HEATING DEMAND", "Q_heating", self.Q_heating, "kW", "Eq. (18)"),
            ("Cooling demand, sensible", "m*cp*(T_pasteur - T_out)", self.Q_sensible_cooling, "kW", "Eq. (19), 1st term"),
            ("Heat losses during cooling", "(1/t) sum A*U*(T_i - T_amb)", self.Q_loss_cooling, "kW", "Eq. (19), 2nd term (subtracted)"),
            ("COOLING DEMAND", "Q_cooling", self.Q_cooling, "kW", "Eq. (19)"),
            ("Sum heating + cooling (gross)", "Q_heating + Q_cooling", self.Q_heating + self.Q_cooling, "kW", "= classes.py calculate_total_energy()[2]"),
            ("Heat capacity flow rate", "m*cp", self.CP_flow, "kW/K", "-"),
        ]
        return pd.DataFrame(rows, columns=["Quantity", "Symbol", "Value", "Unit", "Manuscript ref."])

    def stream_table(self) -> pd.DataFrame:
        """Stream data of the two product streams (hand-over to the pinch analysis of the overall system)."""
        T_p, v = self.T_pasteur_eff, self.inputs
        return pd.DataFrame([
            {"stream": "product heating", "type": "cold", "T_supply_degC": v["T_in"], "T_target_degC": T_p,
             "CP_kW_per_K": self.Q_heating / (T_p - v["T_in"]), "Q_kW": self.Q_heating},
            {"stream": "product cooling", "type": "hot", "T_supply_degC": T_p, "T_target_degC": v["T_out"],
             "CP_kW_per_K": self.Q_cooling / (T_p - v["T_out"]), "Q_kW": self.Q_cooling},
        ])


def pasteurization_model(values: dict) -> PasteurizationResult:
    """Energy demand of continuous pasteurization, manuscript Eq. (18) and (19).

    Q_heating = m*cp*(T_pasteur - T_in)  + (1/t) * sum_i A*U*(T_i - T_amb)          (18)
    Q_cooling = m*cp*(T_pasteur - T_out) - (1/t) * sum_i A*U*(T_i - T_amb)          (19)

    Assumption (manuscript does not define T_i): the product temperature in the loss term is the
    mean temperature of the respective stage, i.e. the time average for a linear profile.
    The holding time does not enter the energy model (manuscript, first paragraph of the section).

    CHANGES vs. classes.Pasteurization (see comparison section at the end of the notebook):
      NEW      optional loss term of Eq. (18)/(19), safety margin, input validation, stream table
      CHANGED  cp in kJ/(kg K) instead of J/(kg K); result object instead of tuple
      SAME     Q_heating / Q_cooling without losses are numerically identical
    """
    v = values
    m = v["mass_flow_kg_h"] / 3600.0                     # kg/s
    cp = v["cp"]                                         # kJ/(kg K)
    T_p = v["T_pasteur"] + v["T_safety_margin"]
    T_in, T_out = v["T_in"], v["T_out"]
    A, U, T_amb = v["loss_area"], v["U_value"], v["T_amb"]

    if m <= 0 or cp <= 0:
        raise ValueError("mass flow and cp must be > 0")
    if not T_in < T_p:
        raise ValueError(f"T_in ({T_in}) must be below the pasteurization temperature ({T_p}).")
    if not T_out < T_p:
        raise ValueError(f"T_out ({T_out}) must be below the pasteurization temperature ({T_p}).")

    CP_flow = m * cp                                     # kW/K  (kg/s * kJ/(kg K))

    # Eq. (18)
    Q_sens_h = CP_flow * (T_p - T_in)
    Q_loss_h = A * U * (0.5 * (T_in + T_p) - T_amb) / 1000.0       # W -> kW
    Q_heating = Q_sens_h + Q_loss_h
    # Eq. (19)
    Q_sens_c = CP_flow * (T_p - T_out)
    Q_loss_c = A * U * (0.5 * (T_p + T_out) - T_amb) / 1000.0
    Q_cooling = Q_sens_c - Q_loss_c

    warns = []
    if Q_cooling <= 0:
        raise ValueError("Cooling demand <= 0: heat losses exceed the sensible cooling load.")
    if A > 0 and T_out < T_amb - 1e-9:
        warns.append("T_out is below ambient temperature: loss term in Eq. (19) becomes negative (heat gain).")

    return PasteurizationResult(
        inputs=dict(v), T_pasteur_eff=T_p,
        Q_sensible_heating=Q_sens_h, Q_loss_heating=Q_loss_h, Q_heating=Q_heating,
        Q_sensible_cooling=Q_sens_c, Q_loss_cooling=Q_loss_c, Q_cooling=Q_cooling,
        CP_flow=CP_flow, warnings=warns,
    )

## 3  Visualisierung (Bedarfsaufteilung, Stromdaten, Sensitivität, P&ID)

In [ ]:
def plot_demand_breakdown(res: PasteurizationResult):
    """Terms of Eq. (18) and (19): sensible term, loss term (signed as in the equation) and total."""
    fig, ax = plt.subplots(figsize=(8, 4.2))
    cats = ["Sensible term\nm*cp*dT", "Loss term\n(signed as in Eq.)", "Total demand"]
    heating = [res.Q_sensible_heating, res.Q_loss_heating, res.Q_heating]
    cooling = [res.Q_sensible_cooling, -res.Q_loss_cooling, res.Q_cooling]
    x, w = np.arange(len(cats)), 0.38
    b1 = ax.bar(x - w / 2, heating, w, color="#d9822b", label="Heating, Eq. (18)")
    b2 = ax.bar(x + w / 2, cooling, w, color="#4c78a8", label="Cooling, Eq. (19)")
    for bars in (b1, b2):
        for b in bars:
            h = b.get_height()
            ax.annotate(f"{h:,.1f}", (b.get_x() + b.get_width() / 2, h), ha="center",
                        va="bottom" if h >= 0 else "top", fontsize=9, xytext=(0, 2 if h >= 0 else -2),
                        textcoords="offset points")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.set(ylabel="Thermal power [kW]", title="Energy demand of the pasteurization process")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


def plot_stream_data(res: PasteurizationResult):
    """Temperature vs. heat flow of the heating and cooling demand (both starting at 0 kW)."""
    v = res.inputs
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot([0, res.Q_heating], [v["T_in"], res.T_pasteur_eff], "b-o", label="Product heating (cold stream)")
    ax.plot([0, res.Q_cooling], [v["T_out"], res.T_pasteur_eff], "r-o", label="Product cooling (hot stream)")
    ax.set(xlabel="Heat flow [kW]", ylabel="Temperature [degC]", title="Stream data of the pasteurization process")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


def pasteurization_sweep(values: dict, param: str, grid) -> pd.DataFrame:
    rows = []
    for x in grid:
        try:
            r = pasteurization_model({**values, param: float(x)})
            rows.append({param: x, "Q_heating": r.Q_heating, "Q_cooling": r.Q_cooling})
        except ValueError:
            continue
    return pd.DataFrame(rows)


def plot_sensitivity(res: PasteurizationResult):
    """One-at-a-time sensitivity of Q_heating and Q_cooling."""
    v = res.inputs
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sweeps = [("T_pasteur", np.linspace(v["T_pasteur"] - 15, v["T_pasteur"] + 15, 31), "Pasteurization temperature [degC]"),
              ("mass_flow_kg_h", np.linspace(0.5, 1.5, 31) * v["mass_flow_kg_h"], "Mass flow rate [kg/h]")]
    for ax, (param, grid, xlabel) in zip(axes, sweeps):
        df = pasteurization_sweep(v, param, grid)
        ax.plot(df[param], df["Q_heating"], label="Q_heating (Eq. 18)")
        ax.plot(df[param], df["Q_cooling"], label="Q_cooling (Eq. 19)")
        ax.axvline(v[param], color="grey", ls="--", lw=0.8)
        ax.set(xlabel=xlabel, ylabel="Thermal power [kW]")
        ax.legend(fontsize=8)
    fig.suptitle("Sensitivity (one parameter varied, all others fixed)")
    fig.tight_layout()
    return fig

In [ ]:
def draw_pid(res: PasteurizationResult):
    """Simplified P&ID of the pasteurization line (ISA-5.1 style tags). Duties come from the demand model."""
    v = res.inputs
    fig, ax = plt.subplots(figsize=(16, 6.6))
    ax.set_xlim(0, 20); ax.set_ylim(0, 7.2); ax.set_aspect("equal"); ax.axis("off")
    LW, y0 = 1.8, 3.5                # line width, y of the product line

    def line(xs, ys, ls="-", lw=LW, color="k"):
        ax.plot(xs, ys, ls=ls, lw=lw, color=color, solid_capstyle="butt", zorder=2)

    def arrow(x0, y0_, x1, y1_):
        ax.annotate("", xy=(x1, y1_), xytext=(x0, y0_), zorder=3,
                    arrowprops=dict(arrowstyle="-|>", lw=LW, color="k", mutation_scale=14))

    def zig(x0, x1, y, amp=0.16, n=8):
        xs = np.linspace(x0, x1, 2 * n + 1)
        ys = y + amp * np.array([0] + [(-1) ** i for i in range(2 * n - 1)] + [0])
        ax.plot(xs, ys, color="k", lw=1.3, zorder=4)

    def bubble(x, y, tag, num, temp=None, r=0.42, text_above=True):
        ax.add_patch(Circle((x, y), r, fc="white", ec="k", lw=1.4, zorder=5))
        ax.text(x, y + 0.1, tag, ha="center", va="center", fontsize=8.5, fontweight="bold", zorder=6)
        ax.text(x, y - 0.16, num, ha="center", va="center", fontsize=8, zorder=6)
        if temp is not None:
            ty = y + r + 0.25 if text_above else y - r - 0.25
            ax.text(x, ty, temp, ha="center", va="center", fontsize=9, color="firebrick", fontweight="bold")

    def valve(x, y, side=+1):
        for sgn in (+1, -1):
            ax.add_patch(Polygon([(x - .25, y + sgn * .22), (x + .25, y + sgn * .22), (x, y)],
                                 fc="white", ec="k", lw=1.4, zorder=5))
        ax.plot([x, x + side * 0.45], [y, y], color="k", lw=1.2, zorder=5)
        ax.add_patch(Circle((x + side * 0.65, y), 0.2, fc="white", ec="k", lw=1.2, zorder=5))

    # ---- equipment -----------------------------------------------------------------------
    ax.add_patch(FancyBboxPatch((0.4, 2.5), 1.8, 2.0, boxstyle="round,pad=0.02,rounding_size=0.3",
                                fc="#eef3f8", ec="k", lw=1.6, zorder=3))
    ax.text(1.3, y0, "T-101\nFeed tank", ha="center", va="center", fontsize=9)
    ax.add_patch(Circle((4.0, y0), 0.45, fc="white", ec="k", lw=1.6, zorder=4))
    ax.add_patch(Polygon([(3.8, y0 - .28), (3.8, y0 + .28), (4.3, y0)], fc="k", zorder=5))
    ax.text(4.0, y0 - 0.8, "P-101", ha="center", fontsize=9)
    ax.add_patch(Circle((7.5, y0), 0.6, fc="white", ec="k", lw=1.6, zorder=4)); zig(7.05, 7.95, y0, 0.15, 3)
    ax.text(7.5, y0 + 0.95, f"E-102 Heater\nQ_heating = {res.Q_heating:,.1f} kW", ha="center", fontsize=9)
    ax.add_patch(FancyBboxPatch((9.8, y0 - 0.3), 3.0, 0.6, boxstyle="round,pad=0.0,rounding_size=0.3",
                                fc="white", ec="k", lw=1.6, zorder=4))
    zig(10.2, 12.4, y0, 0.14, 6)
    ax.text(11.3, y0 + 0.7, f"HT-101 Holding tube\nT = {res.T_pasteur_eff:.1f} degC", ha="center", fontsize=9)
    ax.add_patch(Circle((15.0, y0), 0.6, fc="white", ec="k", lw=1.6, zorder=4)); zig(14.55, 15.45, y0, 0.15, 3)
    ax.text(15.0, y0 - 1.0, f"E-103 Cooler\nQ_cooling = {res.Q_cooling:,.1f} kW", ha="center", fontsize=9)
    ax.add_patch(Rectangle((18.0, y0 - 0.6), 1.8, 1.2, fc="#eef3f8", ec="k", lw=1.6, zorder=3))
    ax.text(18.9, y0, "To filling /\nstorage", ha="center", va="center", fontsize=8.5)

    # ---- product line ---------------------------------------------------------------------
    line([2.2, 3.55], [y0, y0]); line([4.45, 6.9], [y0, y0]); line([8.1, 9.8], [y0, y0])
    line([12.8, 14.4], [y0, y0]); line([15.6, 18.0], [y0, y0])
    arrow(4.7, y0, 5.1, y0); arrow(8.6, y0, 9.0, y0); arrow(13.3, y0, 13.7, y0); arrow(16.3, y0, 16.7, y0)

    # ---- heater utility (from below) with control valve TV-101 ----------------------------
    line([8.0, 8.0], [0.5, y0 - 0.6]); line([7.0, 7.0], [0.5, y0 - 0.6])
    valve(8.0, 1.5, side=+1); ax.text(8.0, 1.95, "TV-101", ha="left", fontsize=8)
    ax.text(8.0, 0.25, "Hot utility in", ha="center", fontsize=8); ax.text(7.0, 0.25, "return", ha="center", fontsize=8)
    line([9.6, 9.6], [1.92, y0], ls=":", lw=1.1)
    bubble(9.6, 1.5, "TIC", "101", f"{res.T_pasteur_eff:.1f} degC", text_above=False)
    line([8.85, 9.18], [1.5, 1.5], ls="--", lw=1.1)

    # ---- cooler utility (from above) with control valve TV-102 ----------------------------
    line([15.5, 15.5], [y0 + 0.6, 6.4]); line([14.5, 14.5], [y0 + 0.6, 6.4])
    valve(15.5, 5.2, side=+1); ax.text(15.5, 5.65, "TV-102", ha="left", fontsize=8)
    ax.text(15.5, 6.65, "Cold utility in", ha="center", fontsize=8); ax.text(14.5, 6.65, "return", ha="center", fontsize=8)
    line([17.1, 17.1], [4.78, y0], ls=":", lw=1.1)
    bubble(17.1, 5.2, "TIC", "102", f"{v['T_out']:.1f} degC")
    line([16.35, 16.68], [5.2, 5.2], ls="--", lw=1.1)

    # ---- inlet instruments ------------------------------------------------------------------
    line([2.9, 2.9], [y0, y0 + 0.58], ls=":", lw=1.1)
    bubble(2.9, y0 + 1.0, "TI", "101", f"{v['T_in']:.1f} degC")
    line([5.3, 5.3], [y0, y0 + 0.58], ls=":", lw=1.1)
    bubble(5.3, y0 + 1.0, "FIC", "101", f"{v['mass_flow_kg_h']:,.0f} kg/h")

    ax.text(0.3, 0.1, "Simplified P&ID (ISA-5.1 style tags), generated from the demand model - not for construction. "
            "Heat recovery is not part of this model.\n"
            f"m = {v['mass_flow_kg_h']:,.0f} kg/h, cp = {v['cp']:.2f} kJ/(kg K)", fontsize=8.5, color="dimgrey", va="bottom")
    ax.set_title("Pasteurization - simplified P&ID", fontsize=12)
    return fig

## 4  Prozesse registrieren
Pasteurisierung ist implementiert; die übrigen Prozesse aus Fig. 1 des Manuscripts sind als *planned* angelegt (inkl. Gleichungsverweisen und Pflicht-Inputs laut Manuscript).

In [ ]:
register_process(ProcessModel(
    key="pasteurization", name="Pasteurization",
    process_type="Thermal activation of fluid products", category="Continuous heating of fluids",
    status="implemented", manuscript_refs="Eq. (18), (19), Fig. 7",
    inputs=PASTEURIZATION_INPUTS, run=pasteurization_model, derive_defaults=pasteurization_derive_defaults,
    figures=[("Demand breakdown (Eq. 18 / 19)", plot_demand_breakdown),
             ("Stream data", plot_stream_data),
             ("Sensitivity analysis", plot_sensitivity),
             ("Simplified P&ID", draw_pid)],
))


def _planned(key, name, ptype, cat, refs, text):
    register_process(ProcessModel(key=key, name=name, process_type=ptype, category=cat,
                                  status="planned", manuscript_refs=refs, planned_inputs_text=text))


_planned("convective_drying", "Convective drying", "Drying", "Convective drying", "Eq. (3)-(9), Fig. 3",
         "- Mass flow rate m_product of the product\n- Initial w1 and final w2 moisture content\n"
         "- Mass flow rate m_air of drying air\n- Initial T0, drying air temperature T1, absolute humidity x0, x1\n"
         "Existing code: classes.SprayDryerModel / AirConditioner / AirCooler (not yet compared)")
_planned("conductive_drying", "Conductive drying", "Drying", "Conductive drying", "Eq. (9)-(11), Fig. 4",
         "- Mass flow rate m_product of product\n- Initial w_in and final w_out moisture content\n"
         "- Drying surface temperature T_surface\n- Mass flow rate m_air of air and inlet conditions T1, x1\n"
         "Existing code: classes.ConductiveDrying (see comparison section: hard-coded 1200 J/kgK and 85 K)")
_planned("autoclaving", "Autoclaving / batch sterilization", "Thermal activation of solid products", "Autoclaving",
         "Eq. (12)-(14), Fig. 5",
         "- Mass of product, packaging, autoclave\n- Specific heat capacities of each component\n"
         "- Initial T_start and target T_steri, T_final temperatures\n- Process times (heating, holding, cooling)\n"
         "- Utility mass flow and inlet temperature\n"
         "Existing code: classes.BatchSterilizationModel (see comparison section: hard-coded -24 K, no loss terms)")
_planned("bath_heating", "Continuous bath heating", "Thermal activation of solid products", "Continuous bath heating",
         "Eq. (15)-(17), Fig. 6",
         "- Mass, cp and inlet temperature of the product\n- Bath target temperature T_bath\n"
         "- Surface area of the bath and insulation characteristics U_bath\n- Ambient temperature and heat transfer coefficient alpha")
_planned("distillation", "Distillation", "Thermal separation", "Thermal phase change", "Eq. (20), (21), Fig. 8",
         "- Mass flow, cp,product1, cp,product2 and inlet temperature of the feed\n- Boiling temperature T_v\n"
         "- Mass fraction w of the evaporated component\n- Latent heat of vaporization h_v\n"
         "- Outlet temperature T_out\n- Inlet/outlet temperature of the utility stream\n"
         "Existing code: classes.DistillationEnergyDemand (to be compared with Eq. 20/21)")

## 5  Rechnen ohne Oberfläche
`run_process` prüft die Inputs, füllt Defaults und rechnet; `report` zeigt Tabellen und Grafiken.

In [ ]:
def _show(obj):
    try:
        from IPython.display import display
        display(obj)
    except ImportError:
        print(obj)


def _heading(text):
    try:
        from IPython.display import display, Markdown
        display(Markdown(text))
    except ImportError:
        print("\n" + text)


def run_process(key: str, **inputs):
    """Run a registered process model. Returns (result, audit_table)."""
    model = PROCESS_REGISTRY[key]
    if model.status != "implemented":
        raise NotImplementedError(f"Process '{model.name}' is planned but not implemented yet.")
    values, audit = resolve_inputs(model, inputs)
    return model.run(values), audit


def report(key: str, result, audit: pd.DataFrame, show_figures: bool = True):
    """Print input audit, result table and figures."""
    model = PROCESS_REGISTRY[key]
    _heading(f"### {model.name} - inputs (required / optional, source of each value)")
    _show(audit)
    for w in getattr(result, "warnings", []):
        print("WARNING:", w)
    _heading("### Results")
    _show(result.summary())
    if hasattr(result, "stream_table"):
        _heading("### Stream data (hand-over to the pinch analysis)")
        _show(result.stream_table())
    if show_figures:
        for title, fn in model.figures:
            _heading(f"### {title}")
            fig = fn(result)
            plt.show()
            plt.close(fig)

In [ ]:
# Minimal call: only the REQUIRED inputs. Everything else falls back to defaults.
# Values of the main.py example: 3000 L/h * 1.045 kg/L, cp = 3890 J/(kg K) -> 3.89 kJ/(kg K) (= default of 'Juice (generic)')
result, audit = run_process("pasteurization", mass_flow_kg_h=3000 * 1.045, T_in=4, T_pasteur=90, T_out=4)
report("pasteurization", result, audit)

## 6  Validierung gegen das Manuscript
Vergleich mit den Zahlen aus *Validation of Process Demand Models*.

**Auffälligkeiten im Manuscript (Pasteurisierung):**
1. **456 kg/h passt nicht zu 291 kW.** Mit den Werten aus `main.py` (3000 l/h · 1,045 kg/l = **3135 kg/h**, cp = 3890 J/kgK, 4 → 90 → 4 °C) ergibt sich genau 291,3 kW. Der Massenstrom im Manuscript-Text ist sehr wahrscheinlich falsch.
2. Im Validierungsabschnitt steht außerdem 0,13 kg/s (= 468 kg/h) *und* 8,33 kg/s (vermutlich aus dem Papiertrocknungsfall kopiert).
3. $T_{out}$ fehlt im Text; 4 °C bestätigt `main.py`.
4. Die „relative deviation“ im Manuscript bezieht sich auf den Modellwert (12,5/291,3 = 4,29 %), nicht auf den Referenzwert.

In [ ]:
# Reference values printed in the manuscript (validation section)
REF = dict(Q_ref=278.8, Q_model=291.3)

# Case A: inputs exactly as printed in the manuscript (456 kg/h). T_out = 4 degC, cp = 3.89 as in main.py.
case_A, _ = run_process("pasteurization", mass_flow_kg_h=456, T_in=4, T_pasteur=90, T_out=4, cp=3.89)

# Case B: inputs of the main.py example (3000 L/h * 1.045 kg/L = 3135 kg/h, cp = 3890 J/(kg K), 4 -> 90 -> 4 degC)
case_B, _ = run_process("pasteurization", mass_flow_kg_h=3000 * 1.045, T_in=4, T_pasteur=90, T_out=4, cp=3.89)

cmp = pd.DataFrame({
    "Manuscript": [REF["Q_model"], 456.0],
    "Case A (manuscript inputs)": [case_A.Q_heating, 456.0],
    "Case B (main.py inputs)": [case_B.Q_heating, 3135.0],
}, index=["Q_heating [kW]  (Eq. 18)", "Mass flow [kg/h]"])
_show(cmp)

dev = abs(REF["Q_model"] - REF["Q_ref"])
print(f"Manuscript model vs. reference [26]: |{REF['Q_model']} - {REF['Q_ref']}| = {dev:.1f} kW")
print(f"  relative to model value    : {dev / REF['Q_model']:.2%}  (this is what the manuscript prints: 4.29 %)")
print(f"  relative to reference value: {dev / REF['Q_ref']:.2%}")
print(f"Printed 456 kg/h gives only {case_A.Q_heating:.1f} kW; the main.py inputs (3135 kg/h) reproduce "
      f"{case_B.Q_heating:.1f} kW -> the mass flow in the manuscript text looks wrong.")

# ---------------- consistency checks of the implementation -----------------------------------
r, _ = run_process("pasteurization", mass_flow_kg_h=1000, T_in=10, T_pasteur=85, T_out=20, cp=3.9)
assert abs(r.Q_heating - 1000 / 3600 * 3.9 * 75) < 1e-9                        # Eq. (18) without losses
assert abs(r.Q_cooling - 1000 / 3600 * 3.9 * 65) < 1e-9                        # Eq. (19) without losses
r_loss, _ = run_process("pasteurization", mass_flow_kg_h=1000, T_in=10, T_pasteur=85, T_out=20, cp=3.9,
                        loss_area=5, U_value=2)
assert r_loss.Q_heating > r.Q_heating and r_loss.Q_cooling < r.Q_cooling       # losses: + heating, - cooling
print("All consistency checks passed.")

## 7  Vergleich mit dem bestehenden Code (`classes.py` / `main.py`)
**Legende:** 🆕 neu im Notebook · ✏️ geändert · ✅ identisch · ⚠️ Auffälligkeit im bestehenden Code

### Pasteurisierung (`classes.Pasteurization` → `pasteurization_model`)
| Aspekt | `classes.py` | Notebook | |
|---|---|---|---|
| Heiz-/Kühlbedarf ohne Verluste (Eq. 18/19) | `m·cp·ΔT/3600/1000` | `CP_flow·ΔT` | ✅ numerisch identisch (siehe Test unten) |
| Einheit cp | J/(kg K), Pflicht | kJ/(kg K), **optional** (Preset je Produkt) | ✏️ |
| Massenstrom | kg/h (in `main.py`: 3000 l/h · 1,045 → kg/h) | kg/h | ✅ (l/h → kg/h bleibt Aufgabe des Nutzers) |
| Verlustterm aus Eq. (18)/(19) | fehlt | optional (A, U, T_amb), Default 0 | 🆕 |
| Sicherheitsaufschlag auf T_pasteur | fehlt | optional, Default 0 K | 🆕 |
| Eingabeprüfung | keine (T_pasteur < T_start liefert negativen Bedarf) | `ValueError` bei unplausiblen Temperaturen/Massenstrom | 🆕 |
| Rückgabe | Tupel (Heizen, Kühlen, **Summe**) | Ergebnisobjekt + Tabelle; Summe als eigene Zeile | ✏️ |
| Stromdaten für die Pinch-Analyse | – | `result.stream_table()` | 🆕 |
| Pinch / GCC / Wärmerückgewinnung | nicht enthalten | bewusst nicht enthalten (im Gesamtsystem) | ✅ |
| Grafiken, P&ID | keine | 3 Grafiken + P&ID | 🆕 |
| Prozessauswahl, Oberfläche, Registry | keine | ja | 🆕 |
| Sprache | deutsche Kommentare | englisch (wie im Paper) | ✏️ |
| Abhängigkeiten | CoolProp (für Pasteurisierung unnötig) | numpy, pandas, matplotlib | ✏️ |
| Beispielwerte | `main.py`: 3135 kg/h, cp 3890 | Default-Preset „Juice (generic)“ = 3,89 kJ/kgK | ✏️ |

### Auffälligkeiten in den anderen Klassen (noch nicht im Notebook, beim Portieren prüfen)
| Klasse | Auffälligkeit | Manuscript |
|---|---|---|
| `BatchSterilizationModel` | ⚠️ `steri_temp - start_temp - 24` (festes −24 K nur beim Produkt); Kühlenergie wird berechnet, aber nicht zurückgegeben; Verlustfaktor auskommentiert | Eq. (12)–(14): Verluste über A·U·(T−T_amb), Halteverluste Eq. (14), Zeiten t |
| `DistillationEnergyDemand` | Einheiten kg/s und kJ (anders als die Pasteurisierung in J) | Eq. (20), (21) noch nicht abgeglichen |
| `ConductiveDrying` | ⚠️ feste Werte `1200` J/kgK und `85` K in `Q_heat`; Verdampfungsenthalpie über CoolProp | Eq. (9)–(11) noch nicht abgeglichen |
| `SprayDryerModel`, `AirConditioner`, `AirCooler` | nicht verglichen | konvektive Trocknung, Eq. (3)–(9) |

In [ ]:
# ---- verbatim copy of classes.Pasteurization (original comments/docstring kept; CoolProp import not needed) ----
class LegacyPasteurization:
    def __init__(self, m_prod, cp, T_start, T_pasteur, T_final):
        """
        :param m_prod: Massenstrom Produkt (kg/h)
        :param cp: spezifische Waermekapazitaet des Produkts (J/(kg K))
        :param T_start: Starttemperatur des Produkts (degC)
        :param T_pasteur: Pasteurisierungstemperatur (degC)
        :param T_final: Endtemperatur nach Abkuehlung (degC)
        """
        self.m_prod = m_prod
        self.cp = cp
        self.T_start = T_start
        self.T_pasteur = T_pasteur
        self.T_final = T_final

    def calculate_heating_energy(self):
        delta_T_heating = self.T_pasteur - self.T_start
        return self.m_prod * self.cp * delta_T_heating / 3600 / 1000  # J/h -> kW

    def calculate_cooling_energy(self):
        delta_T_cooling = self.T_pasteur - self.T_final
        return self.m_prod * self.cp * delta_T_cooling / 3600 / 1000  # J/h -> kW

    def calculate_total_energy(self):
        Q_heating = self.calculate_heating_energy()
        Q_cooling = self.calculate_cooling_energy()
        return Q_heating, Q_cooling, Q_heating + Q_cooling


# ---- 1) numerical comparison: example of main.py -----------------------------------------------
m_prod, cp_J = 3000 * 1.045, 3890                     # kg/h, J/(kg K)  (main.py)
old = LegacyPasteurization(m_prod, cp_J, 4, 90, 4)
Qh_old, Qc_old, Qs_old = old.calculate_total_energy()
new, _ = run_process("pasteurization", mass_flow_kg_h=m_prod, cp=cp_J / 1000, T_in=4, T_pasteur=90, T_out=4)

num = pd.DataFrame({
    "classes.py": [Qh_old, Qc_old, Qs_old],
    "notebook":   [new.Q_heating, new.Q_cooling, new.Q_heating + new.Q_cooling],
}, index=["Q_heating [kW]", "Q_cooling [kW]", "Sum [kW]"])
num["difference"] = num["notebook"] - num["classes.py"]
num["status"] = np.where(num["difference"].abs() < 1e-9, "identical", "DIFFERENT")
_show(num)

# ---- 2) equivalence on random inputs (without loss term the models must agree) ------------------
rng = np.random.default_rng(1)
for _ in range(500):
    m, cp, Ti, Tp = rng.uniform(100, 8000), rng.uniform(3000, 4200), rng.uniform(0, 30), rng.uniform(65, 140)
    To = rng.uniform(0, Tp - 1)
    o = LegacyPasteurization(m, cp, Ti, Tp, To)
    n, _ = run_process("pasteurization", mass_flow_kg_h=m, cp=cp / 1000, T_in=Ti, T_pasteur=Tp, T_out=To)
    assert abs(o.calculate_heating_energy() - n.Q_heating) < 1e-9
    assert abs(o.calculate_cooling_energy() - n.Q_cooling) < 1e-9
print("500 random cases: Q_heating and Q_cooling identical to classes.Pasteurization (no heat losses).")

# ---- 3) behaviour on invalid input --------------------------------------------------------------
bad = LegacyPasteurization(1000, 3890, 90, 4, 4)       # T_pasteur below T_start
print(f"classes.py with T_pasteur < T_start silently returns Q_heating = {bad.calculate_heating_energy():.1f} kW")
try:
    run_process("pasteurization", mass_flow_kg_h=1000, cp=3.89, T_in=90, T_pasteur=4, T_out=4)
except ValueError as e:
    print("notebook raises ValueError:", e)

## 8  Interaktive Oberfläche
Braucht `ipywidgets` (`pip install ipywidgets`). Schritt 1: Prozess wählen → 2: Pflicht-Inputs → 3: optionale Inputs (Haken „use default“ entfernen zum Überschreiben) → *Calculate*.

In [ ]:
def launch_ui():
    """Interactive form: 1) select process, 2) required inputs, 3) optional inputs, then 'Calculate'."""
    try:
        import ipywidgets as w
        from IPython.display import display, clear_output
    except ImportError:
        print("ipywidgets not installed: pip install ipywidgets  (or use run_process() / report() directly)")
        return

    style = {"description_width": "initial"}
    wide = w.Layout(width="520px")
    options = [(m.name + ("" if m.status == "implemented" else "  (planned)"), k) for k, m in PROCESS_REGISTRY.items()]
    selector = w.Dropdown(options=options, value="pasteurization", description="1) Process:", style=style, layout=wide)
    form, out = w.VBox(), w.Output()
    run_btn = w.Button(description="Calculate", button_style="primary", icon="play")
    fields = {}   # key -> (value widget, "use default" checkbox or None)

    def make_row(s: InputSpec):
        if s.kind == "choice":
            wid = w.Dropdown(options=list(s.choices), value=s.default or s.choices[0],
                             description=s.label, style=style, layout=wide)
        else:
            start = s.example if s.example is not None else (s.default if s.default is not None else 0.0)
            wid = w.FloatText(value=float(start), description=f"{s.label} [{s.unit}]", style=style, layout=wide)
        chk, items = None, [wid]
        if not s.required:
            chk = w.Checkbox(value=True, description="use default", indent=False, layout=w.Layout(width="130px"))
            wid.disabled = True
            chk.observe(lambda ch, wid=wid: setattr(wid, "disabled", ch["new"]), names="value")
            items.append(chk)
        if s.note:
            items.append(w.HTML(f"<span style='color:#666;font-size:90%'>{s.symbol}: {s.note}</span>"))
        return wid, chk, w.HBox(items)

    def build_form(key):
        model = PROCESS_REGISTRY[key]
        fields.clear()
        head = [w.HTML(f"<b>{model.process_type} &rsaquo; {model.category} &rsaquo; {model.name}</b>"
                       f" &nbsp; <i>Manuscript: {model.manuscript_refs}</i>")]
        if model.status != "implemented":
            run_btn.disabled = True
            head.append(w.HTML("<p style='color:#a60'>Not implemented yet. Required inputs according to the manuscript:</p>"
                               f"<pre>{model.planned_inputs_text}</pre>"))
            form.children = head
            return
        run_btn.disabled = False
        req = [w.HTML("<h4>2) Required inputs</h4>")]
        opt = [w.HTML("<h4>3) Optional inputs (default is used while 'use default' is ticked)</h4>")]
        for s in model.inputs:
            wid, chk, row = make_row(s)
            fields[s.key] = (wid, chk)
            (req if s.required else opt).append(row)
        form.children = head + req + opt

    def on_run(_):
        with out:
            clear_output(wait=True)
            vals = {k: (None if (chk is not None and chk.value) else wid.value) for k, (wid, chk) in fields.items()}
            try:
                result, audit = run_process(selector.value, **vals)
            except Exception as exc:                      # show input problems instead of a traceback
                print(f"{type(exc).__name__}: {exc}")
                return
            report(selector.value, result, audit)

    def on_select(change):
        build_form(change["new"])
        out.clear_output()

    selector.observe(on_select, names="value")
    run_btn.on_click(on_run)
    build_form(selector.value)
    display(w.VBox([selector, form, run_btn, out]))


launch_ui()

## Offene Punkte
- Weitere Prozesse: Gleichungen und Pflicht-Inputs sind in der Registry hinterlegt, die Modelle fehlen noch.
- cp-Presets sind typische Werte, **nicht** aus dem Manuscript.
- Das P&ID ist ein vereinfachtes Fließbild mit ISA-Tags, kein Planungsdokument.